# CrossFuse v5 - Stage B: Multimodal Training

Trains the full CrossFuse multimodal model (video / audio / sync / fusion
heads, frame-level auxiliary supervision, paired real-fake batching) on
FakeAVCeleb, **now initialised from the FF++-pretrained CLIP encoder produced
by NB-B1** and trained on a **merged FakeAVCeleb + FF++ manifest**.

### What changed from the original v5 run, and why

v5 scored 0.95 in-domain but 0.58 / 0.64 zero-shot. FakeAVCeleb is ~97%
Wav2Lip - a mouth-region reenactment - so the video head never saw a
full-face swap, which is what Celeb-DF and DFDC actually contain. Two changes
address that directly:

1. **Backbone** - CLIP ViT-L/14 with LayerNorm-only tuning, replacing
   EfficientNet-B4 / ImageNet. See NB-B1 for the evidence.
2. **Training mixture** - FF++'s four swap families are merged into the
   training set. They carry no audio, so `has_audio=0` masks them out of every
   audio-dependent loss and metric, and `train_only=1` keeps them out of
   val/test so the in-domain numbers stay comparable to the v5 baseline.

The audio, sync and fusion heads are unchanged and still train on FakeAVCeleb
alone. The sync head's preregistered negative result stands as reported.

**Before the full run**, run the timing gate below: it measures one *fine-tune* epoch, prints a full-run ETA and sets `TIME_BUDGET_S`, so the run stops cleanly with its best checkpoint before Kaggle's 9 h session cap. Single GPU only (`nn.DataParallel` was ~3x slower for CLIP ViT-L/14).

**Attach:** `crossfuse-v5-lib`, `crops-v5` (NB-A), `ffpp-crops-v5` (NB-A3),
`ffpp-encoder-v5` (NB-B1). Internet ON if the encoder needs to re-fetch CLIP.

**Status (2026-09): ran** (1 seed, single T4). Validation AUC: video 0.9991, audio 0.9971, sync 0.5042 (chance; cut). Publish `/kaggle/working` as `crossfuse-v5-ckpt` and attach it to `C-eval`. Cross-dataset results (DFDC 0.865, Celeb-DF 0.845) come from `C-eval`, not from here.

Audit note (2026-10-05): this is the editable recipe, not an executed training log. The recovered main metrics record 887 training clips and do not establish that the optional FF++ mixture was used. The A4/A14/A15 mixture runs are recorded separately in `results/ablation_attribution_v5.csv`.


In [ ]:
# open_clip_torch's real deps (checked against its PyPI wheel metadata):
# torch, torchvision, regex, ftfy, tqdm, huggingface-hub, safetensors,
# timm>=1.0.17. torch/torchvision/tqdm/huggingface-hub are already on Kaggle's
# base image; installing the rest with --no-deps (timm and safetensors
# included -- neither pins a torch/torchvision version, checked directly
# against their wheel metadata) is what keeps pip from ever touching the
# preinstalled torch/torchvision pairing. Needed here separately from NB-B1 --
# each Kaggle notebook is its own kernel, nothing installed there carries over.
!pip install -q --no-deps open_clip_torch timm safetensors ftfy regex pyyaml

import sys, glob, os
_lib_hits = glob.glob("/kaggle/input/**/crossfuse_v5.py", recursive=True)
assert _lib_hits, ("crossfuse_v5.py not found under /kaggle/input -- attach the "
                   "crossfuse-v5-lib dataset in Add Input, then restart the session.")
sys.path.insert(0, os.path.dirname(_lib_hits[0]))

# Verify before crossfuse_v5 (which imports open_clip at module load time)
# gets pulled in below -- fails fast and clearly instead of ~mid-pipeline.
import torch, torchvision
print(f"torch {torch.__version__} | torchvision {torchvision.__version__}")
from torchvision.models import efficientnet_b0  # noqa: F401
import open_clip  # noqa: F401
print("torchvision.models.efficientnet_b0 -- OK | open_clip -- OK")
print("Using crossfuse_v5 library from:", _lib_hits[0])

import os, glob, csv, json
import numpy as np
import torch

from crossfuse_v5 import (
    CONFIG, run_training_pipeline, evaluate_deterministic, seed_everything,
    merge_manifests,
)

# ---- FakeAVCeleb crops (NB-A): the manifest with video_label/audio_label ----
CROP_DIR = None
for p in glob.glob("/kaggle/input/**/manifest.csv", recursive=True):
    with open(p) as f:
        head = f.readline()
    if "audio_label" in head:
        CROP_DIR = os.path.dirname(p)
        break
assert CROP_DIR, ("FakeAVCeleb manifest.csv not found -- attach the crops-v5 dataset.")
print(f"FakeAVCeleb crops: {CROP_DIR}")

with open(os.path.join(CROP_DIR, "manifest.csv")) as f:
    FAV_ROWS = list(csv.DictReader(f))
for r in FAV_ROWS:
    r["video_label"] = int(r["video_label"]); r["audio_label"] = int(r["audio_label"])
    r["fps"] = float(r["fps"])
print(f"  {len(FAV_ROWS)} rows")

# ---- FF++ crops (NB-A3): optional, but this is the whole point of the change ----
FFPP_DIR, FFPP_ROWS = None, []
for p in glob.glob("/kaggle/input/**/manifest.csv", recursive=True):
    with open(p) as f:
        head = f.readline()
    if "method" in head and "target_id" in head:
        FFPP_DIR = os.path.dirname(p)
        break
if FFPP_DIR:
    with open(os.path.join(FFPP_DIR, "manifest.csv")) as f:
        FFPP_ROWS = list(csv.DictReader(f))
    for r in FFPP_ROWS:
        r["label"] = int(r["label"])
    print(f"FF++ crops: {FFPP_DIR} ({len(FFPP_ROWS)} rows)")
else:
    print("[i] No FF++ mixture attached: training on FakeAVCeleb with the FF++-pretrained encoder.")

In [ ]:
# ---- Backbone + pretrained encoder ----
# The encoder checkpoint records the backbone it was trained as; read it back
# rather than assuming, so a ViT checkpoint can never be loaded into an
# EfficientNet build (load_visual_encoder would raise, but late and opaquely).
_enc = glob.glob("/kaggle/input/**/ffpp_encoder_v5.pth", recursive=True)
CONFIG["PRETRAINED_ENCODER"] = _enc[0] if _enc else None

if CONFIG["PRETRAINED_ENCODER"]:
    _meta = torch.load(CONFIG["PRETRAINED_ENCODER"], map_location="cpu",
                       weights_only=False).get("meta", {})
    CONFIG["BACKBONE"] = _meta.get("backbone", CONFIG["BACKBONE"])
    CONFIG["TRAIN_RES"] = _meta.get("train_res", CONFIG["TRAIN_RES"])
    print(f"FF++ encoder: {CONFIG['PRETRAINED_ENCODER']}")
    print(f"  backbone={CONFIG['BACKBONE']} train_res={CONFIG['TRAIN_RES']} "
          f"ffpp_val_auc={_meta.get('val_auc', float('nan')):.4f}")
    assert _meta.get("crop_size") == CONFIG["CROP_SIZE"], (
        f"encoder was trained on {_meta.get('crop_size')}px crops but CONFIG "
        f"says {CONFIG['CROP_SIZE']} -- the features will not transfer")
else:
    raise RuntimeError("No FF++ encoder found. Run B1 and attach ffpp-encoder-v5; use D2 for no-pretraining controls.")

# Verify both preregistered gates from the encoder's companion score artifact.
_gate_path = os.path.join(os.path.dirname(CONFIG["PRETRAINED_ENCODER"]), "ffpp_zeroshot_v5.json")
assert os.path.exists(_gate_path), "Attach the B1 gate score JSON alongside the encoder checkpoint."
with open(_gate_path) as _f:
    _gates = json.load(_f)
for _dataset, _minimum in (("celebdf", 0.85), ("dfdc", 0.75)):
    _auc = _gates.get(_dataset, {}).get("auc", float("nan"))
    assert np.isfinite(_auc) and _auc >= _minimum, f"B1 gate failed or missing: {_dataset} AUC={_auc}"

# CLIP ViT-L/14 needs far less LR than a from-scratch head, and only its
# LayerNorms move. LR_LN is applied to the backbone's LN params by
# backbone_param_groups when the backbone is a ViT.
if CONFIG["BACKBONE"].startswith("clip_"):
    CONFIG["BATCH_CLIPS"] = 4      # 4 x 12 frames = 48 ViT-L images/step on a T4
    CONFIG["GRAD_ACCUM"] = 3       # keeps the effective batch at the v5 value
print(f"batch {CONFIG['BATCH_CLIPS']} clips x {CONFIG['WINDOW_FRAMES']} frames "
      f"x{CONFIG['GRAD_ACCUM']} accum")

# ---- Multi-GPU: DO NOT ENABLE for clip_vit_l14 -- confirmed on a live 2xT4
# run (2026-09-03) to be ~3x SLOWER than single-GPU (2904s/epoch vs 934.7s),
# consistent across 5 epochs, not a fluke. No hang (that part of the fix
# worked), but nn.DataParallel re-replicates the whole 303M-param backbone
# to the second GPU on every forward call, and BATCH_CLIPS=4 means ~1,400+
# forward calls/epoch -- the replication overhead dominates and swamps any
# parallelism benefit. See enable_multi_gpu()'s docstring in crossfuse_v5.py.
# Forced off here regardless of GPU count until a DDP-based rewrite (not
# nn.DataParallel) is validated at real scale, not just a small probe.
CONFIG["MULTI_GPU"] = False
print(f"MULTI_GPU={CONFIG['MULTI_GPU']} (forced off -- see comment above)")

In [ ]:
# ---- Merged training manifest ----
if FFPP_ROWS:
    MANIFEST = merge_manifests(FAV_ROWS, CROP_DIR, FFPP_ROWS, FFPP_DIR,
                               seed=CONFIG["SEED"], verbose=True)
else:
    MANIFEST = FAV_ROWS
    for r in MANIFEST:
        r["crop_dir"] = CROP_DIR
        r["has_audio"] = 1

# Sanity: val/test must stay pure FakeAVCeleb, or the in-domain table stops
# being comparable to the v5 baseline it is meant to be read against.
n_ff = sum(1 for r in MANIFEST if r.get("corpus") == "ffpp")
print(f"\n{len(MANIFEST)} manifest rows before splitting, {n_ff} of them FF++ (train-only)")

In [ ]:
# ---- Timing gate: measure ONE FINE-TUNE epoch, then set a time budget ----
# The old gate ran a frozen epoch, which skips the backbone backward pass and
# so under-reports the real per-epoch cost. EPOCHS_FROZEN=0 goes straight to
# fine-tune (LayerNorms trainable, gradient checkpointing active).
_timing_cfg = dict(CONFIG, EPOCHS_FROZEN=0, EPOCHS_FINETUNE=1, N_SEEDS=1,
                   TIME_BUDGET_S=None)
import time
t0 = time.time()
_probe_result = run_training_pipeline(MANIFEST, CROP_DIR, _timing_cfg,
                                      tag="timing_probe", seed=CONFIG["SEED"],
                                      verbose=True)
elapsed = time.time() - t0
del _probe_result
torch.cuda.empty_cache()

SESSION_CAP_S = 9 * 3600
SAFETY_S = 1.0 * 3600            # setup, eval, checkpoint upload
n_epochs = CONFIG["EPOCHS_FROZEN"] + CONFIG["EPOCHS_FINETUNE"]
print(f"\nOne fine-tune epoch: {elapsed:.0f}s ({elapsed / 60:.1f} min).")
print(f"Full {n_epochs}-epoch run would take ~{n_epochs * elapsed / 3600:.1f} h "
      f"(patience={CONFIG['PATIENCE']} usually stops it earlier).")

CONFIG["TIME_BUDGET_S"] = SESSION_CAP_S - SAFETY_S - elapsed   # minus this probe
print(f"TIME_BUDGET_S set to {CONFIG['TIME_BUDGET_S'] / 3600:.2f} h; the run "
      f"stops cleanly after its best checkpoint when this is exceeded.")

In [ ]:
# ---- Main run across N_SEEDS ----
seed_results = []
for si in range(CONFIG["N_SEEDS"]):
    s = CONFIG["SEED"] + si
    res = run_training_pipeline(MANIFEST, CROP_DIR, CONFIG, tag="main", seed=s, verbose=True)
    seed_results.append(res)
    if si < CONFIG["N_SEEDS"] - 1:
        del res["model"]
        torch.cuda.empty_cache()

_main_result = seed_results[-1]
model = _main_result["model"]
checkpoint_path = _main_result["checkpoint_path"]
train_idx, val_idx, test_idx = (_main_result["train_idx"], _main_result["val_idx"],
                                _main_result["test_idx"])


def _ms(key):
    vals = [r["best_val"][key] for r in seed_results if not np.isnan(r["best_val"][key])]
    return (float(np.mean(vals)), float(np.std(vals))) if vals else (float("nan"), float("nan"))


print("\n" + "=" * 68)
print(f"  MAIN RUN across {CONFIG['N_SEEDS']} seed(s) -- VALIDATION AUC (mean +/- std)")
print("=" * 68)
for k, name in (("video", "Video head"), ("audio", "Audio head"),
                ("sync", "Sync head "), ("fusion", "Fusion head")):
    m_, s_ = _ms(k)
    print(f"  {name}: {m_:.4f} +/- {s_:.4f}")

best_metrics = {
    "n_seeds": CONFIG["N_SEEDS"],
    "per_seed": [{"seed": r["seed"], "best_epoch": r["best_epoch"],
                  "stop_reason": r["stop_reason"],
                  "n_train": len(r["train_idx"]),
                  "n_train_ffpp": sum(MANIFEST[int(i)].get("corpus") == "ffpp" for i in r["train_idx"]),
                  **{k: r["best_val"][k] for k in ("video", "audio", "sync", "fusion",
                                                   "train_video", "train_sync")}}
                 for r in seed_results],
    "val_video_auc_mean_std": _ms("video"), "val_audio_auc_mean_std": _ms("audio"),
    "val_sync_auc_mean_std": _ms("sync"), "val_fusion_auc_mean_std": _ms("fusion"),
    "n_train": len(train_idx), "n_val": len(val_idx), "n_test": len(test_idx),
    "pretrained_encoder": CONFIG["PRETRAINED_ENCODER"],
    "backbone": CONFIG["BACKBONE"],
    "n_train_ffpp": sum(MANIFEST[int(i)].get("corpus") == "ffpp" for i in train_idx),
    "split_keys": {name: [MANIFEST[int(i)]["key"] for i in indices]
                   for name, indices in (("train", train_idx), ("val", val_idx), ("test", test_idx))},
    "config": dict(_main_result["cfg"]),
}
with open("/kaggle/working/best_metrics_v5.json", "w") as f:
    json.dump(best_metrics, f, indent=2)
print(f"\nCheckpoint: {checkpoint_path}")
print("best_metrics_v5.json written. Save Version, publish /kaggle/working as "
      "'crossfuse-v5-ckpt', attach to NB-C and NB-D.")
